# Post-hoc benchmark: pretrained LRW MS-TCN on the already viewed final test

This notebook evaluates the selected pretrained MS-TCN on the existing final-test
archive. The archive was previously used for the BiGRU benchmark, so these numbers
are a post-hoc benchmark and must not be used to tune the model.

## 1. Connect Drive and restore fixed data/checkpoint

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
%pip -q install gdown

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile

import gdown
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


def load_torch_file(path, device="cpu"):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def find_one_archive(name):
    matches = list(Path("/content/drive").rglob(name))
    if not matches:
        raise FileNotFoundError(
            f"Archive {name!r} was not found in Google Drive. "
            "Upload exactly one copy before running this notebook."
        )
    if len(matches) > 1:
        raise RuntimeError(
            f"Several copies of {name!r} were found: "
            + ", ".join(str(path) for path in matches)
        )
    return matches[0]


def extract_archive(archive_path, target_dir):
    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    target_root = target_dir.resolve()
    with zipfile.ZipFile(archive_path, "r") as archive:
        for member in archive.infolist():
            destination = (target_dir / member.filename).resolve()
            if target_root not in destination.parents and destination != target_root:
                raise RuntimeError(f"Unsafe archive member: {member.filename}")
        archive.extractall(target_dir)


def validate_split(split_dir, data_root, name, vocab):
    required_columns = {
        "clip_id", "clip_path", "word", "speaker_id",
        "start_sec", "end_sec",
    }
    csv_path = split_dir / f"{name}.csv"
    frame = pd.read_csv(csv_path)
    missing = required_columns.difference(frame.columns)
    if missing:
        raise ValueError(f"{csv_path} is missing columns: {sorted(missing)}")
    if frame.empty:
        raise ValueError(f"{csv_path} is empty")
    if frame["clip_id"].duplicated().any():
        raise ValueError(f"{csv_path} contains duplicate clip_id values")
    if frame["clip_path"].duplicated().any():
        raise ValueError(f"{csv_path} contains duplicate clip_path values")
    unknown_words = sorted(set(frame["word"]) - set(vocab))
    if unknown_words:
        raise ValueError(f"{csv_path} contains words outside vocab: {unknown_words}")
    if (frame["end_sec"] <= frame["start_sec"]).any():
        raise ValueError(f"{csv_path} contains non-positive intervals")
    missing_files = [
        str(data_root / relative_path)
        for relative_path in frame["clip_path"]
        if not (data_root / relative_path).is_file()
    ]
    if missing_files:
        preview = ", ".join(missing_files[:3])
        raise FileNotFoundError(
            f"{csv_path} references {len(missing_files)} missing clips; examples: {preview}"
        )
    print(
        f"{name}: {len(frame)} clips | "
        f"{frame['speaker_id'].nunique()} speakers | "
        f"{frame['word'].nunique()} words"
    )
    return frame


def load_feature_cache(path, expected_frame, expected_dim=768):
    if not path.exists():
        raise FileNotFoundError(f"Feature cache is missing: {path}")
    cache = load_torch_file(path)
    features = cache["features"].float()
    expected_shape = (len(expected_frame), expected_dim)
    if tuple(features.shape) != expected_shape:
        raise ValueError(
            f"{path} has shape {tuple(features.shape)}, expected {expected_shape}"
        )
    if "words" in cache and list(cache["words"]) != expected_frame["word"].tolist():
        raise ValueError(f"Word order in {path} does not match train.csv")
    if "clip_ids" in cache and list(cache["clip_ids"]) != expected_frame["clip_id"].tolist():
        raise ValueError(f"clip_id order in {path} does not match train.csv")
    if "clip_ids" not in cache:
        print(f"warning: {path.name} has no clip_ids metadata; word order was checked")
    if not torch.isfinite(features).all():
        raise ValueError(f"Feature cache contains NaN or inf values: {path}")
    print(f"validated cache: {path.name} -> {tuple(features.shape)}")
    return features


def ensure_extracted(archive_path, target_dir):
    signature_path = target_dir / ".archive_signature.json"
    signature = {
        "name": archive_path.name,
        "size": archive_path.stat().st_size,
        "mtime_ns": archive_path.stat().st_mtime_ns,
    }
    if (target_dir / "ru_dataset").exists() and signature_path.exists():
        try:
            if json.loads(signature_path.read_text(encoding="utf-8")) == signature:
                print(f"reusing validated extraction: {target_dir}")
                return
        except json.JSONDecodeError:
            pass
    extract_archive(archive_path, target_dir)
    signature_path.write_text(
        json.dumps(signature, indent=2), encoding="utf-8"
    )
    print(f"extracted: {archive_path.name} -> {target_dir}")


drive_root = Path("/content/drive/MyDrive/lipreading_sem4")
train_extract = Path("/content/lipreading_data_v3")
test_extract = Path("/content/lipreading_final_test_landmark_v3")
output_dir = drive_root / "posthoc_mstcn_final_test_benchmark"
output_dir.mkdir(parents=True, exist_ok=True)

upstream_dir = Path("/content/Lipreading_using_Temporal_Convolutional_Networks")
if not upstream_dir.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/mpc001/Lipreading_using_Temporal_Convolutional_Networks.git",
            str(upstream_dir),
        ],
        check=True,
    )
sys.path.insert(0, str(upstream_dir))

checkpoint_path = drive_root / "upstream_lrw_pretrained" / "lrw_resnet18_mstcn_video.pth"
if not checkpoint_path.exists():
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    downloaded = gdown.download(
        id="1vqMpxZ5LzJjg50HlZdj_QFJGm2gQmDUD",
        output=str(checkpoint_path),
        quiet=False,
    )
    if downloaded is None:
        raise RuntimeError("Official LRW checkpoint download failed.")

train_archive = find_one_archive("colab_lipreading_dataset_v3_train_val.zip")
test_archive = find_one_archive("colab_lipreading_final_test_landmark_v3.zip")
ensure_extracted(train_archive, train_extract)
ensure_extracted(test_archive, test_extract)

train_split = (
    train_extract / "ru_dataset" / "03_splits"
    / "ml_splits_mouth_crops_landmark_dataset_v3" / "speaker_top10"
)
test_split = test_extract / "ru_dataset" / "03_splits" / "final_test_landmark_v3"
vocab = (train_split / "vocab.txt").read_text(encoding="utf-8").splitlines()
test_vocab = (test_split / "vocab.txt").read_text(encoding="utf-8").splitlines()
if test_vocab != vocab:
    raise ValueError("Train and final-test vocabularies differ.")

train_df = validate_split(train_split, train_extract / "ru_dataset", "train", vocab)
test_df = validate_split(test_split, test_extract / "ru_dataset", "test", vocab)
word_to_idx = {word: index for index, word in enumerate(vocab)}
idx_to_word = dict(enumerate(vocab))

feature_root = drive_root / "controlled_lrw_frontend_bigru_dataset_v3"
train_original = load_feature_cache(
    feature_root / "dataset_v3_train_lrw_mstcn_pooled.pt", train_df
)
train_hflip = load_feature_cache(
    feature_root / "dataset_v3_train_lrw_mstcn_hflip_pooled.pt", train_df
)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    raise RuntimeError("Enable a GPU runtime before running this benchmark.")

print("train:", len(train_df), "test:", len(test_df))
print("train speakers:", sorted(train_df["speaker_id"].unique()))
print("test speakers:", sorted(test_df["speaker_id"].unique()))
print("vocab:", vocab)
print("device:", device)
preflight = {
    "train_archive": train_archive.name,
    "test_archive": test_archive.name,
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "train_speakers": sorted(train_df["speaker_id"].unique().tolist()),
    "test_speakers": sorted(test_df["speaker_id"].unique().tolist()),
    "vocabulary": vocab,
    "train_feature_shape": list(train_original.shape),
    "train_hflip_feature_shape": list(train_hflip.shape),
    "checks": [
        "CSV schema and required fields",
        "unique clip_id and clip_path",
        "vocabulary equality",
        "positive time intervals",
        "all referenced videos exist",
        "feature order and finite values",
    ],
}
(output_dir / "posthoc_mstcn_preflight.json").write_text(
    json.dumps(preflight, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("preflight passed and saved to:", output_dir / "posthoc_mstcn_preflight.json")
print("WARNING: final-test metrics are post-hoc and must not guide model tuning.")


## 2. Extract fixed pretrained MS-TCN features for final-test

In [ ]:
import cv2
from lipreading.model import Lipreading


def load_lrw_video(path, num_frames=24, input_size=96, crop_size=88):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise ValueError(f"OpenCV could not open video: {path}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise ValueError(f"Video has no readable frames: {path}")

    frames = []
    frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
    offset = (input_size - crop_size) // 2
    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = cap.read()
        if not ok:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame = cv2.resize(frame, (input_size, input_size))
        frame = frame[offset:offset + crop_size, offset:offset + crop_size]
        frame = frame.astype(np.float32) / 255.0
        frames.append((frame - 0.421) / 0.165)
    cap.release()

    if not frames:
        raise ValueError(f"No frames were decoded: {path}")
    while len(frames) < num_frames:
        frames.append(frames[-1].copy())
    return torch.from_numpy(np.stack(frames)).float().unsqueeze(0)


class TestVideoDataset(Dataset):
    def __len__(self):
        return len(test_df)

    def __getitem__(self, index):
        row = test_df.iloc[index]
        video = load_lrw_video(
            test_extract / "ru_dataset" / row["clip_path"]
        )
        return video, index


test_video_loader = DataLoader(
    TestVideoDataset(),
    batch_size=4,
    shuffle=False,
    num_workers=0,
)

tcn_options = {
    "num_layers": 4,
    "kernel_size": [3, 5, 7],
    "dropout": 0.2,
    "dwpw": False,
    "width_mult": 1,
}
pretrained_model = Lipreading(
    modality="video",
    num_classes=500,
    tcn_options=tcn_options,
    backbone_type="resnet",
    relu_type="swish",
    width_mult=1.0,
    use_boundary=False,
    extract_feats=True,
).to(device)

checkpoint = load_torch_file(checkpoint_path)
state_dict = checkpoint.get("model_state_dict", checkpoint)
pretrained_model.load_state_dict(state_dict, strict=True)
for parameter in pretrained_model.parameters():
    parameter.requires_grad = False
pretrained_model.eval()


def extract_pretrained_tcn_embeddings(loader):
    batches = []
    with torch.inference_mode():
        for batch_index, (videos, _) in enumerate(loader, start=1):
            frontend_features = pretrained_model(
                videos.to(device),
                lengths=[videos.shape[2]] * videos.shape[0],
            )
            if frontend_features.ndim != 3 or frontend_features.shape[-1] != 512:
                raise ValueError(
                    "Unexpected pretrained frontend shape: "
                    f"{tuple(frontend_features.shape)}"
                )
            # Official MS-TCN expects [batch, channels, frames].
            sequence = pretrained_model.tcn.mb_ms_tcn(
                frontend_features.transpose(1, 2)
            )
            pooled = sequence.mean(dim=2)
            batches.append(pooled.cpu())
            if batch_index == 1 or batch_index % 25 == 0:
                print(
                    f"feature batches: {batch_index}/{len(loader)} | "
                    f"shape={tuple(pooled.shape)}"
                )
    return torch.cat(batches)


test_cache_path = output_dir / "final_test_mstcn_pooled_features.pt"
expected_test_clip_ids = test_df["clip_id"].tolist()
test_tcn = None
if test_cache_path.exists():
    test_cache = load_torch_file(test_cache_path)
    cached_clip_ids = test_cache.get("clip_ids")
    if cached_clip_ids == expected_test_clip_ids:
        test_tcn = test_cache["features"].float()
        print("loaded cached test MS-TCN features with matching clip order")
    else:
        print("cached test feature order is stale; rebuilding the cache")

if test_tcn is None:
    test_tcn = extract_pretrained_tcn_embeddings(test_video_loader)
    torch.save(
        {
            "features": test_tcn,
            "clip_ids": expected_test_clip_ids,
            "words": test_df["word"].tolist(),
            "speakers": test_df["speaker_id"].tolist(),
            "test_used_for_training": False,
        },
        test_cache_path,
    )
    print("saved test MS-TCN feature cache")

if tuple(test_tcn.shape) != (len(test_df), 768):
    raise ValueError(f"Unexpected test embedding shape: {tuple(test_tcn.shape)}")
if not torch.isfinite(test_tcn).all():
    raise ValueError("Test embeddings contain NaN or inf values.")
print("test pooled features:", tuple(test_tcn.shape))


## 3. Train a fixed linear head on train v3 and evaluate post-hoc on final-test

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_fscore_support,
    balanced_accuracy_score,
    f1_score,
)


class VectorDataset(Dataset):
    def __init__(self, features, words):
        self.features = features.float()
        self.labels = torch.tensor(
            [word_to_idx[word] for word in words],
            dtype=torch.long,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


class LinearProbe(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.classifier = nn.Linear(input_dim, num_classes)

    def forward(self, features):
        return self.classifier(features)


def metrics_for(preds, labels):
    class_ids = np.arange(len(vocab))
    precision, recall, f1, support = precision_recall_fscore_support(
        labels,
        preds,
        labels=class_ids,
        zero_division=0,
    )
    matrix = confusion_matrix(labels, preds, labels=class_ids)
    return {
        "accuracy": float(np.mean(np.asarray(preds) == np.asarray(labels))),
        "macro_f1": float(f1_score(
            labels, preds, labels=class_ids, average="macro", zero_division=0
        )),
        "balanced_accuracy": float(balanced_accuracy_score(labels, preds)),
        "predicted_classes": int(np.count_nonzero(matrix.sum(axis=0))),
        "class_rows": [
            {
                "class_id": int(class_id),
                "word": vocab[class_id],
                "support": int(support[class_id]),
                "predicted_count": int(matrix[:, class_id].sum()),
                "precision": float(precision[class_id]),
                "recall": float(recall[class_id]),
                "f1": float(f1[class_id]),
            }
            for class_id in class_ids
        ],
        "confusion_matrix": matrix,
    }


train_x = torch.cat([train_original, train_hflip])
train_words = train_df["word"].tolist() * 2
test_y = [word_to_idx[word] for word in test_df["word"]]
runs = []
metrics_by_seed = {}

for seed in [42, 123, 2026]:
    set_seed(seed)
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        VectorDataset(train_x, train_words),
        batch_size=64,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    model = LinearProbe(768, len(vocab)).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-3, weight_decay=1e-4
    )
    criterion = nn.CrossEntropyLoss()

    last_train_loss = None
    last_train_accuracy = None
    for epoch in range(10):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_items = 0
        for features, labels in train_loader:
            features = features.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
            total_correct += int((logits.argmax(1) == labels).sum())
            total_items += len(labels)
        last_train_loss = total_loss / total_items
        last_train_accuracy = total_correct / total_items

    model.eval()
    with torch.inference_mode():
        logits = model(test_tcn.to(device))
        preds = logits.argmax(1).cpu().tolist()
    metrics = metrics_for(preds, test_y)
    metrics_by_seed[seed] = metrics
    run = {
        "seed": seed,
        "final_train_loss": last_train_loss,
        "final_train_accuracy": last_train_accuracy,
        "test_accuracy": metrics["accuracy"],
        "test_macro_f1": metrics["macro_f1"],
        "test_balanced_accuracy": metrics["balanced_accuracy"],
        "predicted_classes": metrics["predicted_classes"],
        "test_clips": len(test_y),
        "test_speakers": int(test_df["speaker_id"].nunique()),
        "epochs": 10,
        "augmentation": "original+hflip",
        "encoder": "official LRW ResNet18 frontend + MS-TCN, frozen",
        "head": "linear 768-to-10",
    }
    runs.append(run)
    pd.DataFrame(
        {
            "sample_index": range(len(test_y)),
            "clip_id": test_df["clip_id"].tolist(),
            "speaker_id": test_df["speaker_id"].tolist(),
            "true_word": [idx_to_word[i] for i in test_y],
            "predicted_word": [idx_to_word[i] for i in preds],
        }
    ).to_csv(
        output_dir / f"posthoc_mstcn_test_predictions_seed{seed}.csv",
        index=False,
    )
    pd.DataFrame(metrics["class_rows"]).to_csv(
        output_dir / f"posthoc_mstcn_test_per_class_seed{seed}.csv",
        index=False,
    )
    print(
        f"seed={seed} | train acc={last_train_accuracy:.4f} | "
        f"test acc={metrics['accuracy']:.4f} | "
        f"macro-F1={metrics['macro_f1']:.4f} | "
        f"balanced-acc={metrics['balanced_accuracy']:.4f} | "
        f"classes={metrics['predicted_classes']}/{len(vocab)}"
    )

runs_df = pd.DataFrame(runs).sort_values("seed").reset_index(drop=True)
runs_df.to_csv(output_dir / "posthoc_mstcn_final_test_runs.csv", index=False)
display(runs_df)


## 4. Save summary and freeze interpretation

In [ ]:
summary = runs_df[
    [
        "test_accuracy",
        "test_macro_f1",
        "test_balanced_accuracy",
        "predicted_classes",
    ]
].agg(["mean", "std"]).T
summary.columns = ["mean", "std"]
summary.reset_index(names="metric").to_csv(
    output_dir / "posthoc_mstcn_final_test_summary.csv",
    index=False,
)

per_class = []
combined_matrix = np.zeros((len(vocab), len(vocab)), dtype=int)
for seed, metrics in metrics_by_seed.items():
    frame = pd.DataFrame(metrics["class_rows"])
    frame["seed"] = seed
    per_class.append(frame)
    combined_matrix += metrics["confusion_matrix"]
pd.concat(per_class, ignore_index=True).groupby("word")[
    ["precision", "recall", "f1"]
].agg(["mean", "std"]).to_csv(
    output_dir / "posthoc_mstcn_final_test_per_class_summary.csv"
)

pd.DataFrame(combined_matrix, index=vocab, columns=vocab).to_csv(
    output_dir / "posthoc_mstcn_final_test_confusion_matrix.csv"
)

protocol = {
    "experiment": "posthoc_mstcn_final_test",
    "encoder": "official LRW pretrained ResNet18 frontend plus MS-TCN",
    "encoder_frozen": True,
    "head": "linear_768_to_10",
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "train_speakers": sorted(train_df["speaker_id"].unique().tolist()),
    "test_speakers": sorted(test_df["speaker_id"].unique().tolist()),
    "vocabulary": vocab,
    "seeds": [42, 123, 2026],
    "epochs": 10,
    "augmentation": "original+hflip",
    "test_used_for_training": False,
    "test_used_for_model_selection": False,
    "test_status": "post-hoc benchmark; final test was viewed in an earlier BiGRU experiment",
}
(output_dir / "posthoc_mstcn_protocol.json").write_text(
    json.dumps(protocol, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(output_dir / "README.md").write_text(
    "Post-hoc MS-TCN benchmark\\n\\n"
    "The encoder is frozen and only a linear 768-to-10 head is trained. "
    "The final test was not used for optimization, but it was viewed in an "
    "earlier experiment, so these metrics are not an untouched model-selection "
    "result. Do not tune future models using these files.\\n",
    encoding="utf-8",
)

print(summary)
print("Results saved to:", output_dir)
print("POST-HOC BENCHMARK COMPLETED. Do not tune the model using these test metrics.")
